In [0]:
%pip install xgboost

In [0]:
%pip install imbalanced-learn

In [0]:
dbutils.library.restartPython()

In [0]:
import imblearn
print(imblearn.__version__)

In [0]:
from pyspark.sql import functions as F

feature_df = spark.table("workspace.default.feature_transactions")

fraud_df = feature_df.filter(F.col("isFraud") == 1)
non_fraud_df = feature_df.filter(F.col("isFraud") == 0).sample(fraction=0.05, seed=42)

training_df = fraud_df.union(non_fraud_df)
print(f"Fraud cases: {fraud_df.count()}")
print(f"Non-fraud sample: {non_fraud_df.count()}")

pdf = training_df.toPandas()
pdf = pdf.fillna(0)   # velocity features mein kuch nulls ho sakte hain edge cases mein
print(f"Total training rows: {len(pdf)}")
pdf.head()

In [0]:


import mlflow
import mlflow.xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score, precision_recall_curve, f1_score
)
from imblearn.over_sampling import SMOTE


class FraudModelTrainer:
    FEATURE_COLUMNS = [
        "amount", "oldbalanceOrg", "newbalanceOrig",
        "oldbalanceDest", "newbalanceDest",
        "orig_balance_ratio", "orig_drained_to_zero",
        "dest_balance_ratio", "orig_txn_count_so_far",
        "is_cash_out", "is_transfer", "is_payment",
        # naye velocity features
        "txn_count_1min", "txn_count_5min", "txn_count_15min",
        "txn_amount_sum_1min", "txn_amount_sum_5min",
        "seconds_since_last_txn", "distinct_merchants_24h",
    ]
    LABEL_COLUMN = "isFraud"

    def __init__(self, experiment_name: str = "/Shared/fraudguard_experiments"):
        mlflow.set_experiment(experiment_name)

    def prepare_data(self, pdf):
        X = pdf[self.FEATURE_COLUMNS]
        y = pdf[self.LABEL_COLUMN]
        return train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    def apply_smote(self, X_train, y_train):
        """
        Minority class (fraud) ko synthetically oversample karta hai.
        sampling_strategy=0.1 => fraud:non-fraud ratio ko 1:10 tak le aana
        (doc ke mutabiq), poora 1:1 nahi (warna overfitting ka risk).
        """
        smote = SMOTE(sampling_strategy=0.1, random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
        return X_resampled, y_resampled

    def find_best_threshold(self, y_test, y_proba):
        """Precision-Recall curve se wo threshold dhoondo jo F1 maximize kare."""
        precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
        f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
        best_idx = f1_scores.argmax()
        return thresholds[best_idx] if best_idx < len(thresholds) else 0.5, f1_scores[best_idx]

    def train(self, pdf):
        X_train, X_test, y_train, y_test = self.prepare_data(pdf)

        print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
        X_train_res, y_train_res = self.apply_smote(X_train, y_train)
        print(f"After SMOTE: {y_train_res.value_counts().to_dict()}")

        scale_pos_weight = (y_train_res == 0).sum() / (y_train_res == 1).sum()

        with mlflow.start_run(run_name="xgboost_fraud_v2_smote") as run:
            model = XGBClassifier(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.1,
                scale_pos_weight=scale_pos_weight,
                eval_metric="aucpr",
                random_state=42,
            )
            model.fit(X_train_res, y_train_res)

            y_proba = model.predict_proba(X_test)[:, 1]
            best_threshold, best_f1 = self.find_best_threshold(y_test, y_proba)
            y_pred_optimal = (y_proba >= best_threshold).astype(int)

            auc = roc_auc_score(y_test, y_proba)

            print(f"Best threshold: {best_threshold:.3f} (F1={best_f1:.3f})")
            print(classification_report(y_test, y_pred_optimal))
            print(f"ROC-AUC: {auc:.4f}")

            mlflow.log_param("n_estimators", 200)
            mlflow.log_param("max_depth", 6)
            mlflow.log_param("smote_sampling_strategy", 0.1)
            mlflow.log_param("scale_pos_weight", round(scale_pos_weight, 2))
            mlflow.log_param("best_threshold", round(float(best_threshold), 3))
            mlflow.log_metric("roc_auc", auc)
            mlflow.log_metric("best_f1", best_f1)
            mlflow.xgboost.log_model(model, "model")

            print(f" Run logged: {run.info.run_id}")
            return model, run.info.run_id, best_threshold


trainer = FraudModelTrainer()
model, run_id, best_threshold = trainer.train(pdf)

In [0]:


import mlflow
from mlflow.models import infer_signature


class ModelRegistrar:
    def __init__(self, catalog: str, schema: str, model_name: str):
        self.full_model_name = f"{catalog}.{schema}.{model_name}"
        mlflow.set_registry_uri("databricks-uc")

    def register(self, model, run_id: str, X_sample, y_pred_sample):
        """Signature ke sath model ko dobara log + register karna."""
        signature = infer_signature(X_sample, y_pred_sample)

        with mlflow.start_run(run_id=run_id):
            mlflow.xgboost.log_model(
                model,
                artifact_path="model_with_signature",
                signature=signature,
                input_example=X_sample.head(3),
            )

        model_uri = f"runs:/{run_id}/model_with_signature"
        registered_model = mlflow.register_model(model_uri, self.full_model_name)

        print(f"✅ Registered: {self.full_model_name}, version {registered_model.version}")
        return registered_model


registrar = ModelRegistrar(
    catalog="workspace",
    schema="default",
    model_name="fraudguard_xgboost",
)

# X_test aur predictions chahiye signature ke liye — trainer se re-derive karte hain
X_sample = pdf[trainer.FEATURE_COLUMNS].head(100)
y_sample_pred = model.predict(X_sample)

registered_model = registrar.register(model, run_id, X_sample, y_sample_pred)